In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

class BiodiversityTextFeatureExtractor:
    """
    Extract features from French field notes for biodiversity prediction
    """
    
    def __init__(self):
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=50,
            ngram_range=(1, 2),
            min_df=2
        )
        
    def extract_structured_features(self, notes):
        """
        Extract domain-specific features from field notes
        """
        features = []
        
        for note in notes:
            feature_dict = {}
            
            # Handle missing or NaN values
            if pd.isna(note) or note == 'nan':
                note = ""
            
            note_lower = str(note).lower()
            
            # 1. DISTURBANCE SCORE (0-5)
            disturbance_keywords = [
                'débroussailleuse', 'tracteur', 'machine', 'travaux',
                'tronçonneuse', 'camping', 'défrichement', 'construction',
                'ouvrier', 'laboure', 'coupe'
            ]
            disturbance_score = sum(1 for kw in disturbance_keywords if kw in note_lower)
            feature_dict['disturbance_score'] = min(disturbance_score, 5)
            
            # 2. HABITAT MODIFICATION PERCENTAGE
            cut_percentage = 0
            # Look for patterns like "50% coupé" or "coupé à 70%"
            cut_match = re.search(r'(\d+)\s*%?\s*(?:coupé|coupe)', note_lower)
            if cut_match:
                cut_percentage = int(cut_match.group(1))
            feature_dict['habitat_modification_pct'] = cut_percentage
            
            # 3. POLLUTION PRESENCE (binary)
            pollution_keywords = ['décharge', 'ordures', 'ordure', 'déchet', 'carcasse']
            feature_dict['pollution_present'] = int(any(kw in note_lower for kw in pollution_keywords))
            
            # 4. WATER QUALITY ISSUES
            water_keywords = ['jacinthe', "laitue d'eau", 'sargasse', 'envahi']
            feature_dict['water_quality_issue'] = int(any(kw in note_lower for kw in water_keywords))
            
            # 5. HUMAN ACTIVITY COUNT
            human_keywords = [
                'camping', 'ouvrier', 'tracteur', 'débroussailleuse',
                'tronçonneuse', 'travaux', 'musique', 'campeur', 'personnes'
            ]
            human_count = sum(1 for kw in human_keywords if kw in note_lower)
            feature_dict['human_activity_count'] = human_count
            
            # 6. WEATHER IMPACT (binary)
            weather_keywords = ['pluie', 'vent', 'brouillard', 'pluvieux']
            feature_dict['weather_impact'] = int(any(kw in note_lower for kw in weather_keywords))
            
            # 7. VEGETATION STATUS (categorical -> numerical)
            if 'haute' in note_lower or 'hauts' in note_lower:
                veg_status = 2  # high
            elif 'basse' in note_lower:
                veg_status = 1  # low
            elif 'coupé' in note_lower or 'coupe' in note_lower:
                veg_status = 0  # cut
            else:
                veg_status = 1.5  # medium/unknown
            feature_dict['vegetation_status'] = veg_status
            
            # 8. INVASIVE SPECIES COUNT
            invasive_keywords = ['iguane', 'mangouste', 'tortue', 'floride']
            invasive_count = sum(1 for kw in invasive_keywords if kw in note_lower)
            # Check for explicit counts (e.g., "5 tortues de Floride")
            count_match = re.search(r'(\d+)\s*tortues?', note_lower)
            if count_match:
                invasive_count = max(invasive_count, int(count_match.group(1)))
            feature_dict['invasive_species_count'] = invasive_count
            
            # 9. NOISE POLLUTION
            noise_keywords = ['bruit', 'musique', 'débroussailleuse', 'tracteur', 'nuisance sonnore']
            feature_dict['noise_pollution'] = int(any(kw in note_lower for kw in noise_keywords))
            
            # 10. HABITAT TYPE (categorical)
            if 'canne' in note_lower:
                habitat = 1  # sugarcane
            elif 'banane' in note_lower:
                habitat = 2  # banana
            elif 'mangrove' in note_lower:
                habitat = 3  # mangrove
            elif 'friche' in note_lower:
                habitat = 4  # fallow
            elif 'forêt' in note_lower or 'bois' in note_lower:
                habitat = 5  # forest
            else:
                habitat = 0  # unknown/other
            feature_dict['habitat_type'] = habitat
            
            # 11. OBSERVER ACCOMPANIED (data quality indicator)
            feature_dict['observer_accompanied'] = int('compagnonnage' in note_lower or 'accompagn' in note_lower)
            
            # 12. SITE ACCESSIBILITY ISSUES
            access_keywords = ['impossible', 'impraticable', 'abandon', 'fermé']
            feature_dict['accessibility_issue'] = int(any(kw in note_lower for kw in access_keywords))
            
            features.append(feature_dict)
        
        return pd.DataFrame(features)
    
    def extract_tfidf_features(self, notes, fit=True):
        """
        Extract TF-IDF features from text
        """
        # Clean notes
        cleaned_notes = [str(note) if not pd.isna(note) and note != 'nan' else "" for note in notes]
        
        if fit:
            tfidf_matrix = self.tfidf_vectorizer.fit_transform(cleaned_notes)
        else:
            tfidf_matrix = self.tfidf_vectorizer.transform(cleaned_notes)
        
        # Convert to DataFrame with feature names
        tfidf_df = pd.DataFrame(
            tfidf_matrix.toarray(),
            columns=[f'tfidf_{i}' for i in range(tfidf_matrix.shape[1])]
        )
        
        return tfidf_df
    
    def create_feature_matrix(self, notes, use_tfidf=True, fit=True):
        """
        Create complete feature matrix combining structured and TF-IDF features
        """
        # Extract structured features
        structured_features = self.extract_structured_features(notes)
        
        if use_tfidf:
            # Extract TF-IDF features
            tfidf_features = self.extract_tfidf_features(notes, fit=fit)
            
            # Combine both
            feature_matrix = pd.concat([structured_features, tfidf_features], axis=1)
        else:
            feature_matrix = structured_features
        
        return feature_matrix


# Example usage and demonstration
def demo_pipeline():
    """
    Demonstration of the complete pipeline
    """
    print("=" * 80)
    print("BIODIVERSITY TEXT FEATURE EXTRACTION PIPELINE")
    print("=" * 80)
    
    # Sample notes from your data
    sample_notes = np.array([
        'nan',
        'Compagnonnage par Beatriz Conde',
        'Banane 100% coupé',
        "Beaucoup d'ordure et carcasse de voiture",
        'Camping de plusiers personnes sur ce site',
        'Végetation très secs',
        'CARCASSE DE VOITURE',
        'Canne 100% coupé',
        'Canne 70% coupé',
        "Aménagement d'un parking",
        'DORTOIRE',
        'Canne à sucre 100% coupé',
        'Canne à sucre 50% coupé',
        'Cannes non coupées',
        'canal nettoyé',
        "canal envahi de jacinthe d'eau",
        'Décharges sauvages entre les points 8 et 9',
        'cannes coupées',
        "Pont envahi de jacinthes d'eau",
        'Présence de chat / Décharge sauvage',
        "Jacinthes d'eau",
        "Mangouste + Jacinthes d'eau",
        'Herbes hautes : friche agricole',
        'Tronçonneuse',
        'Pluie, arrêt et reprise le 20/06/2024',
        'Proximité route',
        "Beaucoup d'abeilles sur le gommier rouge",
        'végétation plus haute',
        'Bruit de débroussailleuse',
        'Ruche',
        'Matinné pluvieuse',
        "DEPOT D'ORDURES",
        'Parcelle sans banane et en labourrage de la terre',
        'Canne coupé à 100%- Compagnonnage par Beatriz Conde',
        "Compagnonnage par Beatriz Conde- Beaucoup de bruit dans l'usine",
        'Compagnonnage par Beatriz Conde- TRACTEUR LABOURE LA TERRE',
        'UN TRACTEUR ATTIRZIT LES BUBIBI',
        'CANNE COUPE A 100%%',
        'Oiseaux en lisière de la mangrove',
        'Mangouste, tortoue de floride',
        '5 tortues de Floride',
        '10 tortues de Floride',
        'IGUANE'
    ])
    
    # Initialize extractor
    extractor = BiodiversityTextFeatureExtractor()
    
    # Extract features
    print("\n1. EXTRACTING STRUCTURED FEATURES...")
    print("-" * 80)
    structured_features = extractor.extract_structured_features(sample_notes)
    print("\nStructured Features Shape:", structured_features.shape)
    print("\nFirst 10 rows of structured features:")
    print(structured_features.head(10))
    
    print("\n\n2. EXTRACTING TF-IDF FEATURES...")
    print("-" * 80)
    tfidf_features = extractor.extract_tfidf_features(sample_notes, fit=True)
    print("\nTF-IDF Features Shape:", tfidf_features.shape)
    print("\nTop TF-IDF terms:")
    if hasattr(extractor.tfidf_vectorizer, 'get_feature_names_out'):
        print(extractor.tfidf_vectorizer.get_feature_names_out()[:20])
    
    print("\n\n3. CREATING COMPLETE FEATURE MATRIX...")
    print("-" * 80)
    complete_features = extractor.create_feature_matrix(sample_notes, use_tfidf=True, fit=True)
    print("\nComplete Feature Matrix Shape:", complete_features.shape)
    print("\nFeature columns:")
    print(complete_features.columns.tolist())
    
    print("\n\n4. FEATURE STATISTICS...")
    print("-" * 80)
    print(structured_features.describe())
    
    return extractor, complete_features


def demo_with_prediction():
    """
    Demonstration with a complete prediction pipeline
    """
    print("\n\n" + "=" * 80)
    print("COMPLETE PREDICTION PIPELINE EXAMPLE")
    print("=" * 80)
    
    # Sample data with notes and biodiversity indicators
    np.random.seed(42)
    
    sample_notes = np.array([
        'Canne 100% coupé',
        'Végétation haute, pas de perturbation',
        "Beaucoup d'ordure et carcasse de voiture",
        'Camping de plusieurs personnes sur ce site',
        'Mangrove intacte',
        'Canne 50% coupé',
        "Décharge sauvage + Jacinthes d'eau",
        'Forêt primaire, calme',
        'Tracteur laboure la terre',
        'Friche agricole, herbes hautes',
        'Banane 70% coupé',
        'Zone protégée, végétation dense',
        'Bruit de débroussailleuse',
        '5 tortues de Floride + pollution',
        'Cannes non coupées, zone tranquille',
        'Construction en cours',
        'Pluie forte',
        'Canal nettoyé',
        "Canal envahi de jacinthe d'eau",
        'IGUANE observé'
    ] * 5)  # Repeat to get 100 samples
    
    # Simulate biodiversity indicators (higher = better biodiversity)
    # These would be your actual indicators: species richness, Shannon index, etc.
    biodiversity_indicator = []
    for note in sample_notes:
        note_lower = str(note).lower()
        score = 50  # base score
        
        # Negative impacts
        if 'coupé' in note_lower or 'coupe' in note_lower:
            score -= 15
        if 'ordure' in note_lower or 'décharge' in note_lower:
            score -= 20
        if 'camping' in note_lower or 'tracteur' in note_lower:
            score -= 10
        if 'jacinthe' in note_lower:
            score -= 10
        if 'construction' in note_lower:
            score -= 15
        
        # Positive impacts
        if 'haute' in note_lower and 'coupé' not in note_lower:
            score += 15
        if 'forêt' in note_lower or 'mangrove' in note_lower:
            score += 20
        if 'protégé' in note_lower:
            score += 25
        if 'dense' in note_lower:
            score += 10
        
        # Add some noise
        score += np.random.normal(0, 5)
        biodiversity_indicator.append(max(0, min(100, score)))
    
    biodiversity_indicator = np.array(biodiversity_indicator)
    
    # Initialize extractor
    extractor = BiodiversityTextFeatureExtractor()
    
    # Create feature matrix
    X = extractor.create_feature_matrix(sample_notes, use_tfidf=True, fit=True)
    y = biodiversity_indicator
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Train model
    print("\nTraining Random Forest model...")
    model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred = model.predict(X_test)
    
    # Evaluation
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"\nModel Performance:")
    print(f"  Mean Squared Error: {mse:.2f}")
    print(f"  R² Score: {r2:.2f}")
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\n\nTop 15 Most Important Features:")
    print(feature_importance.head(15))
    
    # Example predictions
    print("\n\nExample Predictions:")
    print("-" * 80)
    for i in range(min(5, len(X_test))):
        print(f"\nNote: {sample_notes[X_test.index[i]]}")
        print(f"Actual biodiversity score: {y_test.iloc[i]:.2f}")
        print(f"Predicted biodiversity score: {y_pred[i]:.2f}")
    
    return extractor, model, X, y


# HOW TO USE WITH YOUR DATA
def usage_example():
    """
    Example of how to use with your actual dataset
    """
    print("\n\n" + "=" * 80)
    print("HOW TO USE WITH YOUR DATA")
    print("=" * 80)
    
    code_example = """
# Load your data
df = pd.read_csv('your_biodiversity_data.csv')

# Initialize extractor
extractor = BiodiversityTextFeatureExtractor()

# Extract features from your notes column
X = extractor.create_feature_matrix(
    df['notes_column'].values,  # Your notes column
    use_tfidf=True,
    fit=True
)

# Combine with other numerical features if you have them
# X_complete = pd.concat([X, df[['temperature', 'humidity', 'altitude']]], axis=1)

# Your target variable (biodiversity indicator)
y = df['biodiversity_indicator']  # e.g., species_richness, shannon_index

# Split and train
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Try different models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

# Random Forest
rf_model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# XGBoost (if installed)
# xgb_model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1)
# xgb_model.fit(X_train, y_train)
# xgb_pred = xgb_model.predict(X_test)

# Evaluate
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
print(f"R² Score: {r2_score(y_test, rf_pred):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, rf_pred)):.3f}")
print(f"MAE: {mean_absolute_error(y_test, rf_pred):.3f}")
"""
    print(code_example)


# Run demonstrations
if __name__ == "__main__":
    # Demo 1: Feature extraction
    extractor, features = demo_pipeline()
    
    # Demo 2: Complete prediction pipeline
    extractor, model, X, y = demo_with_prediction()
    
    # Show usage example
    usage_example()
    
    print("\n\n" + "=" * 80)
    print("NEXT STEPS:")
    print("=" * 80)
    print("""
1. Load your actual dataset with notes and biodiversity indicators
2. Use the BiodiversityTextFeatureExtractor to create features
3. Combine with any other numerical features you have
4. Try different ML models (Random Forest, XGBoost, Neural Networks)
5. Use cross-validation for robust evaluation
6. Analyze feature importance to understand what factors affect biodiversity
7. Consider ensemble methods combining multiple models
    """)

BIODIVERSITY TEXT FEATURE EXTRACTION PIPELINE

1. EXTRACTING STRUCTURED FEATURES...
--------------------------------------------------------------------------------

Structured Features Shape: (43, 12)

First 10 rows of structured features:
   disturbance_score  habitat_modification_pct  pollution_present  \
0                  0                         0                  0   
1                  0                         0                  0   
2                  0                       100                  0   
3                  0                         0                  1   
4                  1                         0                  0   
5                  0                         0                  0   
6                  0                         0                  1   
7                  0                       100                  0   
8                  0                        70                  0   
9                  0                         0                  0   


AttributeError: 'numpy.ndarray' object has no attribute 'iloc'

In [ ]:
# Your columns might be:
# - 'notes': field notes (text)
# - 'species_richness': number of species observed
# - 'abundance': total bird count
# - 'shannon_index': diversity metric